# Value at Risk

Follow one risk number from market data to a plain-English answer. You read the TimesFM forecast, trace a Monte Carlo path, compare forward-looking and historical VaR, add a stress scenario, and ask the data agent about it.

This notebook is the same walkthrough as [`docs/01-value-at-risk.md`](https://github.com/gCloud-Tech-Showcase/datacloud-market-risk/blob/main/docs/01-value-at-risk.md), with charts. Run the cells in order. Your numbers differ slightly from the guide, because TimesFM does not always return identical intervals for identical input.

## Set up

In [ ]:
import google.auth
import matplotlib.pyplot as plt

_, PROJECT_ID = google.auth.default()
print(f"Project: {PROJECT_ID}")

## Before you start

Check the stress scenarios. A fresh deployment has two, `baseline` and `energy_down_15`. If `tech_down_10` is listed from an earlier run, run the **Reset** section at the end first.

In [ ]:
%%bigquery
SELECT scenario_id, description, sector, shock_pct
FROM `raw.stress_scenarios`
ORDER BY scenario_id

## Step 1: Look at the portfolio

Pulsar Capital, a fictional firm, holds 30 US large-cap stocks across three desks. The prices behind the positions are real.

In [ ]:
%%bigquery
SELECT desk, book, COUNT(*) AS positions, SUM(notional_usd) AS notional_usd
FROM `raw.portfolio_positions`
GROUP BY desk, book
ORDER BY desk, book

## Step 2: Read the TimesFM forecast

One `AI.FORECAST` query forecasts all 30 stocks. This is its raw output for JPMorgan. Step 1 is dated the day after the as-of date, even on a weekend: TimesFM infers a daily step, so read step 1 as the next trading day.

In [ ]:
%%bigquery jpm_forecast
SELECT DATE(forecast_timestamp) AS step_date,
       forecast_value,
       prediction_interval_lower_bound AS lower_99,
       prediction_interval_upper_bound AS upper_99,
       ai_forecast_status
FROM `serving.timesfm_forecast`
WHERE symbol = 'JPM'
ORDER BY forecast_timestamp

In [ ]:
jpm_forecast

In [ ]:
%%bigquery jpm_history
SELECT date, close
FROM `processed.daily_returns`
WHERE symbol = 'JPM'
ORDER BY date DESC
LIMIT 60

In [ ]:
history = jpm_history.sort_values("date")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["date"], history["close"], label="Close (last 60 trading days)")
ax.plot(jpm_forecast["step_date"], jpm_forecast["forecast_value"], label="TimesFM median")
ax.fill_between(jpm_forecast["step_date"], jpm_forecast["lower_99"], jpm_forecast["upper_99"], alpha=0.3, label="99% interval")
ax.set_title("JPM: TimesFM forecast and its 99% interval")
ax.set_ylabel("Price (USD)")
ax.legend()
plt.show()

The simulation needs a volatility, not a price. `serving.timesfm_forward_params` turns the 99% interval for step 1 into a 1-day volatility: `ln(upper / lower) / (2 × 2.5758)`.

In [ ]:
%%bigquery vols
SELECT symbol, sector,
       ROUND(100 * timesfm_vol_annualized, 1) AS timesfm_vol_pct,
       ROUND(100 * historical_vol_annualized, 1) AS historical_vol_pct
FROM `serving.timesfm_forward_params`
ORDER BY sector, symbol

In [ ]:
vols.set_index("symbol")[["historical_vol_pct", "timesfm_vol_pct"]].plot.bar(
    figsize=(12, 4), title="Annualized volatility: trailing year vs TimesFM forward view")
plt.ylabel("%")
plt.show()

For most stocks, TimesFM's band implies lower volatility than the trailing year. The demo uses the band as a volatility input. It does not claim the forward view is more accurate.

## Step 3: Trace one Monte Carlo path

Each of the 100,000 paths replays one of the last 1,000 trading days for all 30 stocks together, rescaled to today's TimesFM volatility. The paths are a smoothed bootstrap of those 1,000 days: the count makes the distribution smooth, not better informed. Path 42 replays one day. Its portfolio P&L is the sum of the 30 rows.

In [ ]:
%%bigquery path_42
SELECT historical_date, symbol, desk, notional_usd,
       ROUND(sim_log_return, 5) AS sim_log_return, ROUND(sim_pnl_usd, 2) AS sim_pnl_usd
FROM `serving.mc_simulated_pnl`
WHERE scenario_id = 'baseline' AND path_id = 42
ORDER BY symbol

In [ ]:
print(f"Path 42 replays {path_42['historical_date'].iloc[0]}. Portfolio P&L: ${path_42['sim_pnl_usd'].sum():,.0f}")
path_42.head(10)

## Step 4: Compare forward-looking and historical VaR

VaR at 99% is the loss the portfolio exceeds on only 1 path in 100. Expected Shortfall (ES) is the average loss on those worst paths.

In [ ]:
%%bigquery
SELECT scenario_id, method,
       ROUND(var_95_usd) AS var_95_usd, ROUND(var_99_usd) AS var_99_usd,
       ROUND(es_99_usd) AS es_99_usd, ROUND(var_99_pct_notional, 2) AS var_99_pct
FROM `serving.portfolio_var_summary`
ORDER BY method, scenario_id

The distribution behind the baseline Monte Carlo numbers, with the 99% VaR and ES cut-offs marked:

In [ ]:
%%bigquery baseline_paths
SELECT path_id, SUM(sim_pnl_usd) AS pnl_usd
FROM `serving.mc_simulated_pnl`
WHERE scenario_id = 'baseline'
GROUP BY path_id

In [ ]:
pnl = baseline_paths["pnl_usd"]
var_99 = -pnl.quantile(0.01)
es_99 = -pnl[pnl <= -var_99].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pnl / 1e6, bins=200)
ax.axvline(-var_99 / 1e6, color="red", label=f"99% VaR: ${var_99 / 1e6:.2f}M")
ax.axvline(-es_99 / 1e6, color="black", linestyle="--", label=f"99% ES: ${es_99 / 1e6:.2f}M")
ax.set_title("Simulated 1-day portfolio P&L, 100,000 paths (baseline)")
ax.set_xlabel("P&L (USD millions)")
ax.set_ylabel("Paths")
ax.legend()
plt.show()

Historical VaR is higher. The days that drive it:

In [ ]:
%%bigquery
SELECT date, ROUND(pnl_usd) AS pnl_usd, worst_day_rank
FROM `serving.historical_pnl`
ORDER BY worst_day_rank
LIMIT 5

Two things separate the numbers. The historical view uses the last 500 trading days and the simulation draws from the last 1,000, so the April 2025 tariff selloff weighs more in the historical tail. That window difference is most of the gap. The simulation also rescales each day to TimesFM's volatility, which accounts for a smaller part.

## Step 5: Add a stress scenario and simulate it

Ask a what-if: Information Technology falls 10% tomorrow. The next cell adds the scenario and runs the simulation procedure for it, writing 3,000,000 rows. If the scenario already exists, it is replaced.

In [ ]:
%%bigquery
DELETE FROM `raw.stress_scenarios` WHERE scenario_id = 'tech_down_10';

INSERT INTO `raw.stress_scenarios` (scenario_id, description, sector, shock_pct)
VALUES ('tech_down_10', 'Information Technology falls 10%', 'Information Technology', -0.10);

CALL `serving.simulate_scenario`('tech_down_10');

In [ ]:
%%bigquery by_desk
SELECT scenario_id, desk, ROUND(var_99_usd) AS var_99_usd
FROM `serving.var_by_desk`
WHERE scenario_id IN ('baseline', 'tech_down_10')
ORDER BY desk, scenario_id

In [ ]:
d = by_desk.pivot(index="desk", columns="scenario_id", values="var_99_usd")
(d / 1e6).plot.bar(figsize=(8, 4), title="99% VaR by desk: baseline vs Tech falls 10%")
plt.ylabel("USD millions")
plt.xticks(rotation=0)
plt.show()
d

Only Growth Equities moves, because it holds all the Information Technology stocks. GOOGL is Communication Services under GICS, so this shock does not hit it. The other desks are unchanged to the dollar: every scenario replays the same draws on the same paths.

The shock is applied on top of every path, so this is VaR given the shock: baseline risk plus the shock's loss. It is not Basel stressed VaR, which recalibrates to a stressed period.

## Step 6: Ask the data agent

The data agent answers in plain English and shows the SQL it ran. You can also chat with it in the console under **BigQuery > Agents > Pulsar Capital market risk**. This cell asks it through the Conversational Analytics API.

In [ ]:
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from IPython.display import Markdown, display

QUESTION = "How does the tech_down_10 scenario change our 99% VaR by desk?"

credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
session = AuthorizedSession(credentials)
API = f"https://geminidataanalytics.googleapis.com/v1/projects/{PROJECT_ID}/locations/global"

# OpenTofu gives the agent a new ID whenever its definition changes, so find it
# by its display name.
agents = session.get(f"{API}/dataAgents").json().get("dataAgents", [])
agent = next(a["name"] for a in agents if a.get("displayName") == "Pulsar Capital market risk")

response = session.post(
    f"{API}:chat",
    json={
        "messages": [{"userMessage": {"text": QUESTION}}],
        "dataAgentContext": {"dataAgent": agent},
    },
    timeout=300,
)
response.raise_for_status()

# The agent returns its SQL as generatedSql when it writes a new query. When it
# starts from a verified query, it returns only the BigQuery job it ran, so the
# SQL is read from that job.
sql, answer = [], []
for message in response.json():
    system = message.get("systemMessage", {})
    data = system.get("data", {})
    if data.get("generatedSql"):
        sql.append(data["generatedSql"])
    if data.get("bigQueryJob"):
        job = data["bigQueryJob"]
        sql.append(bigquery.Client(project=PROJECT_ID).get_job(job["jobId"], location=job.get("location")).query)
    if system.get("text", {}).get("textType") == "FINAL_RESPONSE":
        answer.extend(system["text"].get("parts", []))

print("SQL the agent ran:")
print(sql[-1] if sql else "(none returned)")
display(Markdown("**Answer:** " + " ".join(answer)))

## Reset

Remove the scenario you added, so the next run starts from the seeded `baseline` and `energy_down_15`.

In [ ]:
%%bigquery
DELETE FROM `serving.mc_simulated_pnl` WHERE scenario_id = 'tech_down_10';
DELETE FROM `raw.stress_scenarios` WHERE scenario_id = 'tech_down_10';